## Güncel Deney ve Submission Sonuçları

## Güncel Deney ve Submission Sonuçları

| Deney | Açıklama | Model / Yöntem | CV / OOF RMSE | CV Std | Public Score | Durum |
|---|---|---|---:|---:|---:|---|
| Baseline | Median imputation + `Bilinmiyor` categorical imputation + OneHotEncoder | RandomForestRegressor | 1.287300 | 0.011187 | - | Sadece CV |
| Deney 2 | Baseline + numeric missing indicator | RandomForestRegressor | 1.287227 | 0.010851 | - | Sadece CV |
| Deney 3 | Deney 2 + `uyku_oncesi_kafein_mg` ve `uyku_oncesi_ekran_suresi_dk` için `log1p` | RandomForestRegressor | 1.287185 | 0.010841 | - | Sadece CV |
| Deney 4 | Deney 3 + feature engineering v1 | RandomForestRegressor | 1.279990 | 0.011050 | - | Sadece CV |
| Deney 5 | Feature engineering v1 + HGB | HistGradientBoostingRegressor | 1.228947 | 0.008645 | - | Sadece CV |
| Deney 6 | HGB farklı hyperparameter denemesi | HistGradientBoostingRegressor | 1.229513 | 0.009059 | - | Sadece CV |
| Deney 7 | HGB sade tuning + feature engineering v1 + clipped prediction | HistGradientBoostingRegressor | 1.228572 | 0.009463 | 1.21491 | Submission 1 |
| Deney 8 | CatBoost + feature engineering v1 + native categorical handling + clipped prediction | CatBoostRegressor | 1.216780 | 0.010027 | 1.20543 | Submission 2 |
| Deney 9 | CatBoost farklı tuning: `iterations=2000`, `learning_rate=0.025`, `depth=7`, `l2_leaf_reg=7` | CatBoostRegressor | 1.216963 | 0.010279 | - | Submit edilmedi |
| Deney 10 | `0.8 CatBoost + 0.2 HGB` weighted blend + clipped prediction | Weighted Ensemble | - | - | 1.20462 | Submission 3 |
| Deney 11 | OOF optimized `0.844 CatBoost + 0.156 HGB` blend | Weighted Ensemble | 1.216401 | - | - | Submit edilmedi / aday |
| Deney 12 | LightGBM OOF + feature engineering v1 | LGBMRegressor | 1.231275 | - | - | Sadece OOF |
| Deney 13 | XGBoost OOF + feature engineering v1 | XGBRegressor | 1.231147 | - | - | Sadece OOF |
| Deney 14 | OOF optimized `0.814 CatBoost + 0.117 HGB + 0.068 LightGBM + 0.000 XGBoost` blend | Weighted Ensemble | 1.216339 | - | - | Submit edilmedi / aday |
| Ek Deneme - KNN | Feature engineering v1 + scaling + OneHotEncoder | KNeighborsRegressor | 1.438029 | - | - | Başarısız / ensemble’a eklenmedi |
| Deney 15 | CatBoost seed ensemble `42`, `2024`, `3407` + feature engineering v1 | CatBoostRegressor Seed Ensemble | 1.216103 | - | - | Submit edilmedi / aday |
| Deney 16 | OOF optimized `0.856 CatBoost Seed Ensemble + 0.119 HGB + 0.025 LightGBM` blend | Weighted Ensemble | **1.215848** | - | **1.20303** | Submission 4 / Şu an en iyi public |
| Deney 17A | CatBoost seed ensemble `42`, `2024`, `3407` + feature engineering v2 | CatBoostRegressor Seed Ensemble | 1.216825 | - | - | Kötüleşti / submit edilmedi |
| Deney 17B | HGB + feature engineering v2 | HistGradientBoostingRegressor | 1.230043 | - | - | Kötüleşti |
| Deney 17B | LightGBM + feature engineering v2 | LGBMRegressor | 1.231181 | - | - | Kötüleşti |
| Deney 17C | OOF optimized `0.859 CatBoost Seed Ensemble v2 + 0.079 HGB v2 + 0.062 LightGBM v2` blend | Weighted Ensemble | 1.216550 | - | - | Deney 16’dan kötü / submit edilmedi |
| Deney 18 | CatBoost 5 seed ensemble `42`, `2024`, `3407`, `777`, `999` + feature engineering v1 | CatBoostRegressor Seed Ensemble | 1.216003 | - | - | Submit edilmedi / aday |
| Deney 19 | OOF optimized `0.862 CatBoost 5 Seed Ensemble + 0.114 HGB + 0.024 LightGBM` blend | Weighted Ensemble | **1.215783** | - | Beklemede | Yeni en iyi OOF / Submission adayı |


In [12]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

print("Train:", X.shape)
print("Test:", X_test.shape)
print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Train: (56000, 22)
Test: (24000, 22)
Numeric columns: 15
Categorical columns: 7


In [13]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [11]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [1.2816447  1.28348368 1.27256982 1.29351856 1.30528642]
Mean RMSE: 1.287300635393191
Std RMSE: 0.011186657457849115


## Deney 2 - Median Imputation + Missing Indicator

In [5]:
numeric_transformer_exp2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_transformer_exp2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_exp2, num_cols),
        ("cat", categorical_transformer_exp2, cat_cols)
    ]
)

model_exp2 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline_exp2 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp2),
    ("model", model_exp2)
])

scores_exp2 = cross_val_score(
    pipeline_exp2,
    X,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp2 = -scores_exp2

print("Fold RMSE:", rmse_scores_exp2)
print("Mean RMSE:", rmse_scores_exp2.mean())
print("Std RMSE:", rmse_scores_exp2.std())

Fold RMSE: [1.28208303 1.28363382 1.27253663 1.29350328 1.30437916]
Mean RMSE: 1.2872271839573095
Std RMSE: 0.010851417954733506


## Deney 3 - Log Transform + Missing Indicator

In [6]:
X_exp3 = X.copy()
X_test_exp3 = X_test.copy()

log_cols = [
    "uyku_oncesi_kafein_mg",
    "uyku_oncesi_ekran_suresi_dk"
]

for col in log_cols:
    if col in X_exp3.columns:
        X_exp3[col] = np.log1p(X_exp3[col])
        X_test_exp3[col] = np.log1p(X_test_exp3[col])

numeric_transformer_exp3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_transformer_exp3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_exp3 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_exp3, num_cols),
        ("cat", categorical_transformer_exp3, cat_cols)
    ]
)

model_exp3 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline_exp3 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp3),
    ("model", model_exp3)
])

scores_exp3 = cross_val_score(
    pipeline_exp3,
    X_exp3,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp3 = -scores_exp3

print("Fold RMSE:", rmse_scores_exp3)
print("Mean RMSE:", rmse_scores_exp3.mean())
print("Std RMSE:", rmse_scores_exp3.std())

Fold RMSE: [1.28203055 1.28351529 1.27253212 1.29357373 1.30427411]
Mean RMSE: 1.287185160843363
Std RMSE: 0.010840623409594944


## Deney 4 - Feature Engineering + Log Transform + Missing Indicator

In [14]:
X_exp4 = X.copy()
X_test_exp4 = X_test.copy()

# Log dönüşümü
log_cols = [
    "uyku_oncesi_kafein_mg",
    "uyku_oncesi_ekran_suresi_dk"
]

for col in log_cols:
    if col in X_exp4.columns:
        X_exp4[col] = np.log1p(X_exp4[col])
        X_test_exp4[col] = np.log1p(X_test_exp4[col])


def add_features(df):
    df = df.copy()

    # Stres ve çalışma yükü
    if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
        df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

    # Uyku kalitesi göstergesi
    if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
        df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

    # Uyku bozulma skoru
    if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
        df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

    # Dijital yük
    if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
        df["dijital_kafein_yuku"] = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]

    # Aktivite / stres dengesi
    if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
        df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

    return df


X_exp4 = add_features(X_exp4)
X_test_exp4 = add_features(X_test_exp4)

num_cols_exp4 = X_exp4.select_dtypes(include=np.number).columns.tolist()
cat_cols_exp4 = X_exp4.select_dtypes(include="object").columns.tolist()

numeric_transformer_exp4 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_transformer_exp4 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_exp4 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_exp4, num_cols_exp4),
        ("cat", categorical_transformer_exp4, cat_cols_exp4)
    ]
)

model_exp4 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline_exp4 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp4)
])

scores_exp4 = cross_val_score(
    pipeline_exp4,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp4 = -scores_exp4

print("Fold RMSE:", rmse_scores_exp4)
print("Mean RMSE:", rmse_scores_exp4.mean())
print("Std RMSE:", rmse_scores_exp4.std())

Fold RMSE: [1.27703181 1.27639545 1.26436099 1.28405915 1.29810131]
Mean RMSE: 1.2799897415709276
Std RMSE: 0.011050026622092155


## Deney 5 - Feature Engineering + HistGradientBoostingRegressor

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model_exp5 = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=42
)

pipeline_exp5 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp5)
])

scores_exp5 = cross_val_score(
    pipeline_exp5,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp5 = -scores_exp5

print("Fold RMSE:", rmse_scores_exp5)
print("Mean RMSE:", rmse_scores_exp5.mean())
print("Std RMSE:", rmse_scores_exp5.std())

Fold RMSE: [1.22842344 1.22863866 1.21706492 1.22660698 1.24400267]
Mean RMSE: 1.2289473334773597
Std RMSE: 0.008645253496770222


## Deney 6 - HistGradientBoosting Hyperparameter Denemesi

In [ ]:
model_exp6 = HistGradientBoostingRegressor(
    max_iter=800,
    learning_rate=0.03,
    max_leaf_nodes=45,
    min_samples_leaf=20,
    l2_regularization=0.05,
    random_state=42
)

pipeline_exp6 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp6)
])

scores_exp6 = cross_val_score(
    pipeline_exp6,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp6 = -scores_exp6

print("Fold RMSE:", rmse_scores_exp6)
print("Mean RMSE:", rmse_scores_exp6.mean())
print("Std RMSE:", rmse_scores_exp6.std())

Fold RMSE: [1.22964944 1.22862504 1.21702951 1.22701673 1.24524317]
Mean RMSE: 1.2295127784213464
Std RMSE: 0.009058856132983253


## Deney 7 - HistGradientBoosting daha sade tuning

In [ ]:
model_exp7 = HistGradientBoostingRegressor(
    max_iter=700,
    learning_rate=0.04,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42
)

pipeline_exp7 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp7)
])

scores_exp7 = cross_val_score(
    pipeline_exp7,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp7 = -scores_exp7

print("Fold RMSE:", rmse_scores_exp7)
print("Mean RMSE:", rmse_scores_exp7.mean())
print("Std RMSE:", rmse_scores_exp7.std())

Fold RMSE: [1.22849956 1.2290441  1.21597071 1.22432274 1.24502272]
Mean RMSE: 1.228571966718488
Std RMSE: 0.00946258977464743


## Deney 7 ile Submission Üretimi

In [ ]:
best_pipeline = pipeline_exp7

best_pipeline.fit(X_exp4, y)

test_preds = best_pipeline.predict(X_test_exp4)

submission = pd.DataFrame({
    "id": test_ids,
    TARGET: test_preds
})

submission.head()

submission_clipped = submission.copy()

submission_clipped[TARGET] = submission_clipped[TARGET].clip(0, 10)

print(submission_clipped.shape)
print(submission_clipped.head())
print(submission_clipped[TARGET].describe())

submission.to_csv("submission_exp7_hgb.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   5.837882
1   2                   6.839400
2   3                   3.182555
3   4                   7.189681
4   5                   3.609035
count    24000.000000
mean         5.939861
std          1.850152
min          0.132316
25%          4.635788
50%          6.031417
75%          7.339575
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


## Submission Result

Best local model: Deney 7 - HistGradientBoostingRegressor with feature engineering  
CV Mean RMSE: 1.228572  
CV Std: 0.009463  
Public Score: 1.21491  

The public score is slightly better than the cross-validation score, which suggests that the validation setup is reasonably consistent and the model generalizes well to the public test split.

## Deney 8 - CatBoostRegressor + Feature Engineering

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

print("Train:", X.shape)
print("Test:", X_test.shape)

Train: (56000, 22)
Test: (24000, 22)


In [3]:
X_exp4 = X.copy()
X_test_exp4 = X_test.copy()

log_cols = [
    "uyku_oncesi_kafein_mg",
    "uyku_oncesi_ekran_suresi_dk"
]

for col in log_cols:
    if col in X_exp4.columns:
        X_exp4[col] = np.log1p(X_exp4[col])
        X_test_exp4[col] = np.log1p(X_test_exp4[col])


def add_features(df):
    df = df.copy()

    if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
        df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

    if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
        df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

    if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
        df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

    if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
        df["dijital_kafein_yuku"] = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]

    if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
        df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

    return df


X_exp4 = add_features(X_exp4)
X_test_exp4 = add_features(X_test_exp4)

print("X_exp4:", X_exp4.shape)
print("X_test_exp4:", X_test_exp4.shape)

X_exp4: (56000, 27)
X_test_exp4: (24000, 27)


In [4]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

X_cb = X_exp4.copy()
X_test_cb = X_test_exp4.copy()

cat_cols_cb = X_cb.select_dtypes(include="object").columns.tolist()

for col in cat_cols_cb:
    X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
    X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols_cb]

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores_exp8 = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
    X_train_fold = X_cb.iloc[train_idx]
    X_val_fold = X_cb.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model_cb = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=1500,
        learning_rate=0.035,
        depth=6,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=200,
        early_stopping_rounds=100
    )

    model_cb.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_features_idx,
        eval_set=(X_val_fold, y_val_fold),
        use_best_model=True
    )

    val_pred = model_cb.predict(X_val_fold)
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred))

    rmse_scores_exp8.append(fold_rmse)

    print(f"Fold {fold} RMSE:", fold_rmse)

rmse_scores_exp8 = np.array(rmse_scores_exp8)

print("Fold RMSE:", rmse_scores_exp8)
print("Mean RMSE:", rmse_scores_exp8.mean())
print("Std RMSE:", rmse_scores_exp8.std())

0:	learn: 2.1899899	test: 2.2002100	best: 2.2002100 (0)	total: 55.2ms	remaining: 1m 22s
200:	learn: 1.2203253	test: 1.2387517	best: 1.2387517 (200)	total: 1.89s	remaining: 12.2s
400:	learn: 1.1993460	test: 1.2270202	best: 1.2270202 (400)	total: 3.93s	remaining: 10.8s
600:	learn: 1.1871140	test: 1.2239233	best: 1.2238886 (596)	total: 5.36s	remaining: 8.02s
800:	learn: 1.1768064	test: 1.2228847	best: 1.2228847 (800)	total: 6.83s	remaining: 5.96s
1000:	learn: 1.1675482	test: 1.2223728	best: 1.2223562 (997)	total: 8.33s	remaining: 4.15s
1200:	learn: 1.1580040	test: 1.2223533	best: 1.2222052 (1149)	total: 9.83s	remaining: 2.45s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.222205227
bestIteration = 1149

Shrink model to first 1150 iterations.
Fold 1 RMSE: 1.2222052278876236
0:	learn: 2.1933248	test: 2.1844791	best: 2.1844791 (0)	total: 8.78ms	remaining: 13.2s
200:	learn: 1.2219388	test: 1.2329316	best: 1.2329316 (200)	total: 1.53s	remaining: 9.9s
400:	learn: 1.2002286

In [5]:
final_catboost = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=1500,
    learning_rate=0.035,
    depth=6,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=200
)

final_catboost.fit(
    X_cb,
    y,
    cat_features=cat_features_idx
)

catboost_preds = final_catboost.predict(X_test_cb)

submission_catboost = pd.DataFrame({
    "id": test_ids,
    TARGET: catboost_preds
})

submission_catboost[TARGET] = submission_catboost[TARGET].clip(0, 10)

print(submission_catboost.shape)
print(submission_catboost.head())
print(submission_catboost[TARGET].describe())

submission_catboost.to_csv("submission_exp8_catboost.csv", index=False)


0:	learn: 2.1911022	total: 6.84ms	remaining: 10.3s
200:	learn: 1.2215720	total: 1.44s	remaining: 9.29s
400:	learn: 1.2020544	total: 2.88s	remaining: 7.91s
600:	learn: 1.1899524	total: 5.36s	remaining: 8.01s
800:	learn: 1.1804349	total: 7.07s	remaining: 6.17s
1000:	learn: 1.1720815	total: 8.69s	remaining: 4.33s
1200:	learn: 1.1637714	total: 10.3s	remaining: 2.57s
1400:	learn: 1.1556201	total: 12s	remaining: 847ms
1499:	learn: 1.1514546	total: 12.8s	remaining: 0us
(24000, 2)
   id  bilissel_performans_skoru
0   1                   5.953550
1   2                   6.662038
2   3                   2.951826
3   4                   7.121280
4   5                   3.659968
count    24000.000000
mean         5.933198
std          1.869631
min          0.000000
25%          4.637116
50%          6.038805
75%          7.320178
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


## Deney 9 - CatBoost daha kontrollü tuning

In [6]:
rmse_scores_exp9 = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
    X_train_fold = X_cb.iloc[train_idx]
    X_val_fold = X_cb.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model_cb_exp9 = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=2000,
        learning_rate=0.025,
        depth=7,
        l2_leaf_reg=7,
        random_seed=42,
        verbose=200,
        early_stopping_rounds=150
    )

    model_cb_exp9.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_features_idx,
        eval_set=(X_val_fold, y_val_fold),
        use_best_model=True
    )

    val_pred = model_cb_exp9.predict(X_val_fold)
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred))

    rmse_scores_exp9.append(fold_rmse)
    print(f"Fold {fold} RMSE:", fold_rmse)

rmse_scores_exp9 = np.array(rmse_scores_exp9)

print("Fold RMSE:", rmse_scores_exp9)
print("Mean RMSE:", rmse_scores_exp9.mean())
print("Std RMSE:", rmse_scores_exp9.std())

0:	learn: 2.1999555	test: 2.2105387	best: 2.2105387 (0)	total: 8.7ms	remaining: 17.4s
200:	learn: 1.2268813	test: 1.2467166	best: 1.2467166 (200)	total: 1.67s	remaining: 15s
400:	learn: 1.2008780	test: 1.2295322	best: 1.2295322 (400)	total: 3.32s	remaining: 13.3s
600:	learn: 1.1887167	test: 1.2258086	best: 1.2258086 (600)	total: 5.76s	remaining: 13.4s
800:	learn: 1.1771890	test: 1.2238568	best: 1.2238552 (795)	total: 7.63s	remaining: 11.4s
1000:	learn: 1.1670465	test: 1.2224648	best: 1.2224634 (999)	total: 9.53s	remaining: 9.51s
1200:	learn: 1.1574943	test: 1.2223745	best: 1.2223467 (1159)	total: 11.4s	remaining: 7.61s
1400:	learn: 1.1480510	test: 1.2220696	best: 1.2220160 (1375)	total: 13.3s	remaining: 5.69s
1600:	learn: 1.1386739	test: 1.2220152	best: 1.2218529 (1531)	total: 15.2s	remaining: 3.78s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 1.221852927
bestIteration = 1531

Shrink model to first 1532 iterations.
Fold 1 RMSE: 1.2218529278684553
0:	learn: 2.20361

## Deney 10 CatBoost + HGB ensemble/blend

In [7]:
sub_hgb = pd.read_csv("submission_exp7_hgb.csv")
sub_cat = pd.read_csv("submission_exp8_catboost.csv")

TARGET = "bilissel_performans_skoru"

for w in [0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    blend = sub_cat.copy()
    blend[TARGET] = w * sub_cat[TARGET] + (1 - w) * sub_hgb[TARGET]
    blend[TARGET] = blend[TARGET].clip(0, 10)

    print(
        f"cat_weight={w}",
        blend[TARGET].describe()[["mean", "std", "min", "max"]].to_dict()
    )

cat_weight=0.6 {'mean': 5.935863123665517, 'std': 1.8592699295766804, 'min': 0.05292637021029836, 'max': 10.0}
cat_weight=0.7 {'mean': 5.935196814297757, 'std': 1.861541603216446, 'min': 0.03969477765772377, 'max': 10.0}
cat_weight=0.75 {'mean': 5.934863659613875, 'std': 1.8627573792386807, 'min': 0.03307898138143647, 'max': 10.0}
cat_weight=0.8 {'mean': 5.9345305049299935, 'std': 1.8640263105492374, 'min': 0.026463185105149174, 'max': 10.0}
cat_weight=0.85 {'mean': 5.934197350246114, 'std': 1.865348288669059, 'min': 0.019847388828861886, 'max': 10.0}
cat_weight=0.9 {'mean': 5.933864195562232, 'std': 1.8667232008979031, 'min': 0.013231592552574587, 'max': 10.0}


In [8]:
blend_weight = 0.8

submission_blend = sub_cat.copy()
submission_blend[TARGET] = (
    blend_weight * sub_cat[TARGET]
    + (1 - blend_weight) * sub_hgb[TARGET]
)

submission_blend[TARGET] = submission_blend[TARGET].clip(0, 10)

print(submission_blend.shape)
print(submission_blend.head())
print(submission_blend[TARGET].describe())

submission_blend.to_csv("submission_exp10_catboost_hgb_blend.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   5.930417
1   2                   6.697511
2   3                   2.997972
3   4                   7.134960
4   5                   3.649781
count    24000.000000
mean         5.934531
std          1.864026
min          0.026463
25%          4.639064
50%          6.037295
75%          7.322082
max         10.000000
Name: bilissel_performans_skoru, dtype: float64
